In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from category_encoders import CountEncoder # pip install category_encoders

# For DBSCAN + UMAP
import umap # pip install umap-learn
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier

# For Autoencoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 2: Load Data and Prevent Leakage
# Load the dataset
df = pd.read_csv(r'data\submissions.csv')

# Separate target immediately to prevent any accidental leakage during feature engineering
X = df.drop(columns=['is_valid'])
y = df['is_valid'].astype(int) # Assuming 0 = fake/anomalous, 1 = genuine

# Perform train-test split BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

In [ ]:
# Cell 3: Custom Preprocessing Function
def engineer_features(data):
    df_engineered = data.copy()
    
    # 1. Monetary Normalization
    df_engineered['exchangeRate'] = df_engineered['exchangeRate'].fillna(1.0)
    
    monetary_cols = [
        'baseSalary', 'totalCompensation', 'firstYearTotalCompensation', 
        'avgAnnualStockGrantValue', 'totalStockGrantValue', 'avgAnnualBonusValue'
    ]
    for col in monetary_cols:
        df_engineered[col] = pd.to_numeric(df_engineered[col], errors='coerce').fillna(0)
        df_engineered[f'{col}_normalized'] = df_engineered[col] * df_engineered['exchangeRate']
    
    # 2. Date Handling
    df_engineered['offerDate'] = pd.to_datetime(df_engineered['offerDate'], errors='coerce', utc=True)
    df_engineered['submittedAt'] = pd.to_datetime(df_engineered['submittedAt'], errors='coerce', utc=True)
    df_engineered['days_to_submit'] = (df_engineered['submittedAt'] - df_engineered['offerDate']).dt.days
    
    # 3. Numeric Conversions for behavioral features
    df_engineered['userAccountAgeDays'] = pd.to_numeric(df_engineered['userAccountAgeDays'], errors='coerce')
    df_engineered['userPriorSubmissions'] = pd.to_numeric(df_engineered['userPriorSubmissions'], errors='coerce')
    
    # 4. Drop useless, redundant, and complex non-numeric columns
    cols_to_drop = monetary_cols + [
        'uuid', 'offerDate', 'submittedAt', 'exchangeRate', 'baseSalaryCurrency', 
        'stockGrantCurrency', 'bonusCurrency', 'userCurrency', 'userExchangeRate',
        'companyInfo_icon', 'companyInfo_name', 'companyInfo_slug', 'companyInfo_registered',
        'tags', 'otherDetails', 'userId', 'threadId', 'commentsAndRepliesCount',
        'vestingSchedule', 'additionalBonuses' # Dropping complex string/JSON structures
    ]
    df_engineered.drop(columns=[col for col in cols_to_drop if col in df_engineered.columns], inplace=True)
    
    return df_engineered

X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)

In [ ]:
# Cell 4: Encoders and Scalers Pipeline

# High-cardinality categorical features (including focusTag)
high_cardinality_cols = [
    'company', 'title', 'jobFamily', 'jobFamilySlug', 'focusTag',
    'location', 'locationSlug', 'ipHash', 'deviceHash', 
    'cityId', 'dmaId', 'countryId'
]
# Retain only those present in dataset
high_cardinality_cols = [col for col in high_cardinality_cols if col in X_train_eng.columns]

# Low-cardinality categorical features
low_cardinality_cols = [
    'level', 'workArrangement', 'compPerspective', 'salaryFormat', 
    'employmentType', 'gender', 'ethnicity', 'education', 
    'stockType', 'companySize', 'fundingStage'
]
low_cardinality_cols = [col for col in low_cardinality_cols if col in X_train_eng.columns]

# Force all remaining object/string columns to string type for safety
categorical_cols = high_cardinality_cols + low_cardinality_cols
for col in categorical_cols:
    X_train_eng[col] = X_train_eng[col].astype(str)
    X_test_eng[col] = X_test_eng[col].astype(str)

# Strictly filter numeric columns using select_dtypes to prevent string contamination
numeric_cols = X_train_eng.select_dtypes(include=[np.number]).columns.tolist()

# 1. Pipeline for Numerical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 2. Pipeline for High Cardinality
high_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', CountEncoder(combine_min_nan_groups=True))
])

# 3. Pipeline for Low Cardinality
low_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('high_card', high_card_transformer, high_cardinality_cols),
        ('low_card', low_card_transformer, low_cardinality_cols)
    ]
)

# Fit strictly on train, transform both
X_train_processed = preprocessor.fit_transform(X_train_eng)
X_test_processed = preprocessor.transform(X_test_eng)

# Convert outputs to float32
X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

print(f"Final feature matrix shape: {X_train_processed.shape}")

In [ ]:
# Cell 5 (DBSCAN): UMAP Dimensionality Reduction
# UMAP is preferred over PCA as it preserves non-linear neighborhood distances

print("Fitting UMAP on Training Data...")
umap_reducer = umap.UMAP(n_components=3, random_state=42, n_neighbors=15, min_dist=0.1)

# Fit and transform train, ONLY transform test
X_train_umap = umap_reducer.fit_transform(X_train_processed)
X_test_umap = umap_reducer.transform(X_test_processed)

print("UMAP transformation complete.")

In [ ]:

# Cell 6 (DBSCAN): Hyperparameter Tuning and Metrics

import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# --------------------------------------------------
# 1. Define hyperparameter search space
# --------------------------------------------------
eps_values = [0.3, 0.5, 0.7, 1.0, 1.2, 1.5, 2.0]
min_samples_values = [5, 10, 15, 20, 25]

best_f1 = -1
best_params = None
best_dbscan = None
best_train_labels = None

results = []

# --------------------------------------------------
# 2. Tune DBSCAN on the training set
# --------------------------------------------------
for eps in eps_values:
    for min_samples in min_samples_values:

        dbscan = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        cluster_labels = dbscan.fit_predict(X_train_umap)

        # Map noise (-1) to Fake (0), other clusters to Valid (1)
        y_pred = np.where(cluster_labels == -1, 0, 1)

        # Evaluate fake-class F1-score
        fake_f1 = f1_score(
            y_train,
            y_pred,
            pos_label=0,
            zero_division=0
        )

        fake_recall = (
            np.mean(y_pred[y_train == 0] == 0)
            if np.sum(y_train == 0) > 0 else 0
        )

        train_accuracy = accuracy_score(y_train, y_pred)
        noise_ratio = np.mean(cluster_labels == -1)

        results.append({
            "eps": eps,
            "min_samples": min_samples,
            "accuracy": train_accuracy,
            "fake_f1": fake_f1,
            "fake_recall": fake_recall,
            "noise_ratio": noise_ratio
        })

        # Select parameters with highest Fake-class F1
        if fake_f1 > best_f1:
            best_f1 = fake_f1
            best_params = {
                "eps": eps,
                "min_samples": min_samples
            }
            best_dbscan = dbscan
            best_train_labels = cluster_labels

# --------------------------------------------------
# 3. Display tuning results
# --------------------------------------------------
results = sorted(
    results,
    key=lambda x: x["fake_f1"],
    reverse=True
)

print("Top 10 DBSCAN Hyperparameter Combinations")
print("-" * 85)
print(
    f"{'eps':<8}{'min_samples':<14}"
    f"{'Accuracy':<12}{'Fake F1':<12}"
    f"{'Fake Recall':<14}{'Noise Ratio':<12}"
)

for result in results[:10]:
    print(
        f"{result['eps']:<8.2f}"
        f"{result['min_samples']:<14}"
        f"{result['accuracy']:<12.4f}"
        f"{result['fake_f1']:<12.4f}"
        f"{result['fake_recall']:<14.4f}"
        f"{result['noise_ratio']:<12.4f}"
    )

print("\nBest Hyperparameters:")
print(f"eps: {best_params['eps']}")
print(f"min_samples: {best_params['min_samples']}")
print(f"Best Training Fake F1-score: {best_f1:.4f}")

# --------------------------------------------------
# 4. Evaluate best DBSCAN configuration on training set
# --------------------------------------------------
y_train_pred_dbscan = np.where(
    best_train_labels == -1, 0, 1
)

print("\n--- Training Set Evaluation (Tuned DBSCAN) ---")
print(f"Accuracy: {accuracy_score(y_train, y_train_pred_dbscan):.4f}")
print(
    f"Fake F1 Score: "
    f"{f1_score(y_train, y_train_pred_dbscan, pos_label=0, zero_division=0):.4f}"
)
print("\nClassification Report:")
print(classification_report(
    y_train,
    y_train_pred_dbscan,
    labels=[0, 1],
    target_names=["Fake (0)", "Valid (1)"],
    zero_division=0
))

# --------------------------------------------------
# 5. Predict test labels using 1-NN on DBSCAN labels
# --------------------------------------------------
knn = KNeighborsClassifier(n_neighbors=1)

knn.fit(X_train_umap, y_train_pred_dbscan)

y_test_pred_dbscan = knn.predict(X_test_umap)

# --------------------------------------------------
# 6. Evaluate test set
# --------------------------------------------------
print("\n--- Test Set Evaluation (KNN via Tuned DBSCAN Labels) ---")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred_dbscan):.4f}")
print(
    f"Fake F1 Score: "
    f"{f1_score(y_test, y_test_pred_dbscan, pos_label=0, zero_division=0):.4f}"
)
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred_dbscan,
    labels=[0, 1],
    target_names=["Fake (0)", "Valid (1)"],
    zero_division=0
))
